# Single neuron example
To understand how a single neuron works, let's consider a simple example.

Imagine you are building the RES in Villa Pennisi. The success of the RES can be measured on a scale from 0 to 1.

We want to figure out which factors are most important for successfully producing a good RES.

We will consider three factors:

1. Number of hours of sleep (average per night)
2. Number of Campari sodas (average per day)
3. Luck (from 0 to 1)

Our goal is to determine the values of the weights that best explain the data!

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

In [2]:
# (Colab) Enable interactive widgets
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

import ipywidgets as widgets
from ipywidgets import interact


In [3]:
# --------------------------
# 1) Create a tiny dataset
# --------------------------
# Think of each RES iteration as two numbers:
#   x1 = number of hours of sleep per night ~ 0..10
#   x2 = number of Campari Soda per day     ~ 0..20
# Target y = "RES performance index"          (0..1)
np.random.seed(7) # reproducibility
n = 80
x1 = np.random.uniform(0, 10, size=n)    # shots on target
x2 = np.random.uniform(0, 20, size=n)    # successful high presses

# Hidden "true" relationship (unknown to students):
true_w1, true_w2, true_b = 0.08, 0.03, 0.20
noise = np.random.normal(0, 0.05, size=n)
y_true = true_w1 * x1 + true_w2 * x2 + true_b + noise
y_true = np.clip(y_true, 0.0, 1.0)

X = np.stack([x1, x2], axis=1)

In [4]:
# --------------------------
# 2) Single neuron function
# --------------------------
def neuron_output(X, w1, w2, b):
    return X[:, 0]*w1 + X[:, 1]*w2 + b

# --------------------------
# 3) Plotting helper
# --------------------------
def plot_neuron(w1, w2, b):
    y_pred = neuron_output(X, w1, w2, b)
    mse = np.mean((y_pred - y_true)**2)

    # Figure with 2 panels
    plt.figure(figsize=(12, 5))

    # ---- Left: "pitch" of inputs with prediction field ----
    ax1 = plt.subplot(1, 2, 1)

    # Background grid of predictions to show what the neuron "believes"
    gx1 = np.linspace(0, 10, 120)
    gx2 = np.linspace(0, 20, 120)
    G1, G2 = np.meshgrid(gx1, gx2)
    G_pred = (G1 * w1 + G2 * w2 + b).clip(0, 1)

    # pcolormesh background
    ax1.pcolormesh(G1, G2, G_pred, shading='auto')
    # Overlay ground-truth points colored by actual performance
    sc = ax1.scatter(x1, x2, c=y_true, edgecolor='k')

    ax1.set_title("Background color = neuron's prediction")
    ax1.set_xlabel("Hours of sleep (avg. per night)")
    ax1.set_ylabel("Number of Campari Soda (avg. per day)")
    cbar = plt.colorbar(sc, ax=ax1)
    cbar.set_label("Actual RES performance (0..1)")

    # ---- Right: Predicted vs Actual ----
    ax2 = plt.subplot(1, 2, 2)
    ax2.scatter(y_true, y_pred, edgecolor='k')
    ax2.plot([0,1],[0,1])  # perfect-fit line
    ax2.set_xlim(0,1); ax2.set_ylim(0,1)
    ax2.set_xlabel("Actual RES performance (0..1)")
    ax2.set_ylabel("Neuron RES prediction (0..1)")
    ax2.set_title(f"Predicted vs Actual   •   ERROR = {mse:.4f}")

    plt.tight_layout()
    plt.show()

In [5]:
# --------------------------
# 4) Interactive sliders
# --------------------------
# Intuition hints you can say out loud:
# - Increase w1: the neuron cares more about shots on target.
# - Increase w2: the neuron cares more about pressing intensity.
# - Increase b: bumps predictions up even with zero inputs (prior optimism).
w1_slider = widgets.FloatSlider(value=0.05, min=-0.2, max=0.2, step=0.005, description='w1 (sleep)')
w2_slider = widgets.FloatSlider(value=0.02, min=-0.2, max=0.2, step=0.005, description='w2 (campari)')
b_slider  = widgets.FloatSlider(value=0.10, min=-0.5, max=0.5,  step=0.01,  description='bias (luck)')

ui = widgets.VBox([w1_slider, w2_slider, b_slider])
out = widgets.interactive_output(plot_neuron, {'w1': w1_slider, 'w2': w2_slider, 'b': b_slider})

display(ui, out)

Output()

# Gradient descent
It is virtually impossible to find the optimal weigths to miminize the error in a real neural network!

How to do that?

We can use the *gradient descent* algorithm.

Let's see how it works on a single neuron.

In [22]:
# Gradient Descent Visualizer — Single Neuron (recover true weights; clip toggle + OLS reference)
# -----------------------------------------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

# --------------------------
# 1) Data generator (same ranges, same seed)
# --------------------------
def make_data(n=80, seed=7, noise_sigma=0.05, clip_targets=False):
    rng = np.random.default_rng(seed)
    x1 = rng.uniform(0, 10, size=n)   # sleep
    x2 = rng.uniform(0, 20, size=n)   # Campari
    true_w1, true_w2, true_b = 0.08, 0.03, 0.20
    y = true_w1*x1 + true_w2*x2 + true_b + rng.normal(0, noise_sigma, size=n)
    if clip_targets:
        y = np.clip(y, 0.0, 1.0)
    X = np.stack([x1, x2], axis=1).astype(np.float64)
    return X, y, (true_w1, true_w2, true_b)

# Closed-form OLS (normal equation) for reference
def ols_solution(X, y):
    # add bias column
    Xd = np.column_stack([X, np.ones(len(X))])
    beta = np.linalg.pinv(Xd.T @ Xd) @ (Xd.T @ y)
    w = beta[:2]
    b = float(beta[2])
    return w, b

# --------------------------
# 2) Model helpers
# --------------------------
def forward(X, w, b):
    return X @ w + b

def mse(y_pred, y):
    e = y_pred - y
    return float(np.mean(e*e))

# --------------------------
# 3) Gradient descent on STANDARDIZED features; map back each epoch
# --------------------------
def train_gd_standardized(X, y, lr=5e-3, epochs=300, init_seed=123):
    mu = X.mean(axis=0)
    sigma = X.std(axis=0) + 1e-12
    Xs = (X - mu) / sigma

    rng = np.random.default_rng(init_seed)
    ws = rng.normal(0, 0.05, size=X.shape[1])  # std-space weights
    bs = rng.normal(0, 0.05)                   # std-space bias (scalar)

    history = {"w": [], "b": [], "loss": [], "y_pred": []}
    XT = Xs.T
    n = len(y)

    for ep in range(epochs + 1):
        # map to original scale
        w = ws / sigma
        b = bs - float(mu @ w)

        y_pred = forward(X, w, b)
        L = mse(y_pred, y)

        history["w"].append(w.copy())
        history["b"].append(float(b))
        history["loss"].append(L)
        history["y_pred"].append(y_pred.copy())

        if ep == epochs:
            break

        # grads in standardized space
        e = (Xs @ ws + bs) - y
        dws = (2.0 / n) * (XT @ e)
        dbs = 2.0 * float(np.mean(e))

        ws -= lr * dws
        bs -= lr * dbs

    return history

# --------------------------
# 4) UI
# --------------------------
seed = widgets.IntSlider(value=7, min=0, max=9999, step=1, description='Seed')
noise = widgets.FloatSlider(value=0.05, min=0.0, max=0.20, step=0.01, description='Noise')
clip_toggle = widgets.ToggleButtons(options=[('No clip (recover true)', False), ('Clip targets [0,1]', True)],
                                    value=False, description='Targets')

lr = widgets.FloatLogSlider(value=5e-3, base=10, min=-4, max=-1, step=0.01, description='LR')
epochs = widgets.IntSlider(value=20, min=20, max=700, step=10, description='Epochs')
init_seed = widgets.IntSlider(value=123, min=0, max=9999, step=1, description='Init w/b seed')

train_button = widgets.Button(description='Train', button_style='primary')
epoch_view = widgets.IntSlider(value=0, min=0, max=0, step=1, description='View epoch')

out_area = widgets.Output()
ui = widgets.VBox([
    widgets.HBox([seed, noise, clip_toggle]),
    widgets.HBox([lr, epochs, init_seed, train_button]),
    epoch_view,
    out_area
])
display(ui)

# --------------------------
# 5) Run & Plot
# --------------------------
X, y, true_params = make_data(seed=seed.value, noise_sigma=noise.value, clip_targets=clip_toggle.value)
history = None

def plot_epoch(hist, ep, X, y, true_params, w_ols, b_ols):
    with out_area:
        clear_output(wait=True)
        y_pred = np.ravel(hist["y_pred"][ep])
        L = float(hist["loss"][ep])
        w = hist["w"][ep]
        b = float(hist["b"][ep])

        plt.figure(figsize=(14,5))

        # Left: parity plot
        ax1 = plt.subplot(1, 2, 1)
        ax1.scatter(y, y_pred, s=28, alpha=0.85, edgecolor='k', linewidths=0.3)
        ax1.plot([min(y.min(), y_pred.min()), max(y.max(), y_pred.max())],
                 [min(y.min(), y_pred.min()), max(y.max(), y_pred.max())],
                 'r--', linewidth=1)
        ax1.set_xlabel("Actual RES success")
        ax1.set_ylabel("Neuron prediction")
        ax1.set_title(f"Predicted vs Actual  |  Epoch {ep}  |  MSE = {L:.4f}")

        # Right: loss + reference line where it ends
        ax2 = plt.subplot(1, 2, 2)
        ax2.plot(hist["loss"])
        ax2.axvline(ep, linestyle="--", color='r')
        ax2.set_xlabel("Epoch")
        ax2.set_ylabel("MSE loss")
        ax2.set_title("Loss during Gradient Descent")

        plt.tight_layout()
        plt.show()

        tw1, tw2, tb = true_params
        print("Current parameters (original scale):")
        print(f"  w_sleep  (x1) = {w[0]: .4f}")
        print(f"  w_campari(x2) = {w[1]: .4f}")
        print(f"  bias          = {b: .4f}")
        print("\nReference values:")
        print(f"  True (data gen):    w1={tw1:.2f}, w2={tw2:.2f}, b={tb:.2f}")
        print(f"  OLS closed-form:    w1={w_ols[0]:.4f}, w2={w_ols[1]:.4f}, b={b_ols:.4f}")

def on_train_clicked(_):
    global X, y, history
    X, y, true_params = make_data(seed=seed.value, noise_sigma=noise.value, clip_targets=clip_toggle.value)
    # Train
    history = train_gd_standardized(X, y, lr=lr.value, epochs=epochs.value, init_seed=init_seed.value)
    # OLS reference on the SAME (X, y)
    w_ols, b_ols = ols_solution(X, y)

    # Show last epoch by default
    epoch_view.max = len(history["loss"]) - 1
    epoch_view.value = epoch_view.max
    plot_epoch(history, epoch_view.value, X, y, true_params, w_ols, b_ols)

def on_epoch_change(change):
    if change['name'] == 'value' and history is not None:
        w_ols, b_ols = ols_solution(X, y)
        plot_epoch(history, change['new'], X, y, true_params=(0.08, 0.03, 0.20), w_ols=w_ols, b_ols=b_ols)

train_button.on_click(on_train_clicked)
epoch_view.observe(on_epoch_change)

print("Tip: keep 'No clip' ON to recover the true weights. If you turn 'Clip targets' ON,")
print("OLS (and thus GD) will fit the clipped data, so weights shift (slopes down, bias up).")


Tip: keep 'No clip' ON to recover the true weights. If you turn 'Clip targets' ON,
OLS (and thus GD) will fit the clipped data, so weights shift (slopes down, bias up).
